# 💻 Notebook do Aluno — Aula 09: Agentes de IA ReAct, tools e function calling

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 09/14 — Módulo 3: Interfaces, Agentes e Integração**  
**⏱️ 1h40min**  
**🤖 ReAct · @tool · AgentExecutor**  
**🔁 Andaime 50%**  

---

## 🎯 Objetivo da aula

Entender o que diferencia um agente de uma chain. Construir um agente com 3 tools que decide autonomamente qual ferramenta usar para cada pergunta — com o loop de raciocínio visível via verbose=True.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime dos exercícios.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-ollama langchain-community duckduckgo-search chromadb pymupdf -q

from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_ollama import ChatOllama
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
llm = ChatOllama(model="gpt-oss:120b", temperature=0)

# Retriever do CKP02 já indexado
db        = Chroma(persist_directory="/content/ckp02", embedding_function=embeddings)
retriever = db.as_retriever(search_kwargs={"k":3})

# 👉 LACUNA 1: escreva a description da tool de busca nos documentos
@tool
def buscar_nos_documentos(query: str) -> str:
    """___"""  # escreva: quando usar, quando NÃO usar, o que retorna
    docs = retriever.invoke(query)
    if not docs: return "Nenhum documento relevante encontrado."
    return "\n\n".join(f"[pág.{d.metadata.get('page',0)+1}] {d.page_content}" for d in docs)

# 👉 LACUNA 2: escreva a description da tool de busca na web
@tool
def buscar_na_web(query: str) -> str:
    """___"""  # quando usar vs. quando NÃO usar (ex: não para docs internos)
    return DuckDuckGoSearchRun().run(query)

# 👉 LACUNA 3: escreva a description da calculadora
@tool
def calcular(expressao: str) -> str:
    """___"""  # mencionar que recebe expressão Python e retorna número
    try: return str(eval(expressao,{"__builtins__":{}},{}))
    except Exception as e: return f"Erro: {e}"

# 👉 LACUNA 4: monte o AgentExecutor com verbose=True e max_iterations=5
agente   = create_react_agent(llm, [buscar_nos_documentos, buscar_na_web, calcular], hub.pull("hwchase17/react"))
executor = AgentExecutor(agent=agente, tools=[___], verbose=___, max_iterations=___, handle_parsing_errors=True)

# Testar com 4 perguntas que exercitam cada cenário
for q in [
    "Qual é a cláusula de garantia no documento?",         # → RAG
    "Qual é o dólar hoje?",                               # → Web
    "Quanto é 450 * 1.12?",                               # → Calc
    "Qual o prazo de garantia em dias (meses × 30)?",     # → RAG + Calc
]:
    print(f"\n{'='*50}\nPergunta: {q}")
    print(executor.invoke({"input":q})["output"])

---

## ✍️ Suas anotações

Registre aqui as observações da aula (qualidade dos resultados, comparações e conclusões do grupo).

---

## 🏋️ Exercícios da Aula 09

Quatro exercícios práticos em sequência — da primeira tool `@tool` ao roster de 4 tools — para montar um agente ReAct que roteia entre os documentos, a web e a calculadora do domínio do grupo.

Grupo 3–4 · Google Colab: valide cada escolha da tool no log `verbose=True` e registre no notebook a análise do roteamento.


### Exercício 1 — Sua primeira tool: @tool + description · ★★☆ · 10 min

*Individual · Colab*

1. Complete o decorator que transforma a função em tool de agente.
2. Escreva a description no padrão da aula: quando usar + o que retorna + quando NÃO usar.
3. Rode e inspecione `name`, `description` e um `.invoke()` isolado da tool.

> **💡 Dica:** o agente só lê o nome e a description — nunca o corpo da função; incluir o "NÃO use para..." é o que separa tools parecidas.


In [ ]:
# Exercício 1 — sua primeira tool de agente
!pip install -q langchain-core langchain-ollama langchain-community chromadb

from langchain_core.tools import tool
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a09e1", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

# 👉 LACUNA 1: decorator que transforma função Python em tool de agente
___
def buscar_nos_documentos(query: str) -> str:
    # 👉 LACUNA 2: description — quando usar + o que retorna + quando NÃO usar
    """___"""
    docs = retriever.invoke(query)
    if not docs:
        return "Nenhum documento relevante encontrado."
    return "\n\n".join(
        f"[pág.{d.metadata.get('page', 0) + 1}] {d.page_content}" for d in docs)

print(buscar_nos_documentos.name, "→", buscar_nos_documentos.description[:80])
print(buscar_nos_documentos.invoke("prazo de garantia")[:200])


### Exercício 2 — Montar o agente: roster de tools + AgentExecutor · ★★☆ · 10 min

*Individual · Colab*

1. Complete o roster com as 3 tools do domínio.
2. Complete o freio do loop no `AgentExecutor` (padrão da aula).
3. Rode a pergunta multi-step e leia no log do `verbose=True` quais tools o agente acionou.

> **💡 Dica:** sem `max_iterations`, um agente confuso pode girar para sempre — e cada iteração reinjeta Thought + Observation no contexto.


In [ ]:
# Exercício 2 — montar o agente: roster + AgentExecutor
!pip install -q langchain langchain-classic langchain-ollama langchain-community chromadb duckduckgo-search

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a09e2", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

prompt_react = hub.pull("hwchase17/react")

@tool
def buscar_nos_documentos(query: str) -> str:
    """Use quando o usuário perguntar sobre informações dos documentos do
    domínio (manuais, contratos, regulamentos). Retorna trechos relevantes
    com número de página. NÃO use para busca na web nem para cálculos."""
    docs = retriever.invoke(query)
    if not docs:
        return "Nenhum documento relevante encontrado."
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

@tool
def buscar_na_web(query: str) -> str:
    """Use quando precisar de informações atuais NÃO presentes nos documentos
    (cotações, notícias, eventos recentes). NÃO use para conteúdo interno
    do domínio do grupo."""
    return DuckDuckGoSearchRun().run(query)

@tool
def calcular(expressao: str) -> str:
    """Use para cálculos matemáticos. Recebe expressão Python válida
    (ex: '24*30'). NÃO use para buscar informações."""
    try:
        return str(eval(expressao, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"


# 👉 LACUNA 1: primeira tool do roster (documentos do domínio)
# 👉 LACUNA 2: segunda tool (busca na web)
# 👉 LACUNA 3: terceira tool (cálculos)
tools = [___, ___, ___]

agente   = create_react_agent(llm, tools, prompt_react)
executor = AgentExecutor(
    agent=agente, tools=tools, verbose=True,
    # 👉 LACUNA 4: freio do loop — padrão da aula
    max_iterations=___,
    handle_parsing_errors=True,
)
print(executor.invoke(
    {"input": "Qual o prazo de garantia em dias (meses × 30)?"}
)["output"])


### Exercício 3 — Limite do loop: max_iterations e intermediate_steps · ★★☆ · 10 min

*Individual · Colab*

1. Complete os dois limites: o padrão da aula (5) e o enxuto (2).
2. Complete o campo que preserva o rascunho das iterações.
3. Compare os `output` finais: em qual executor o agente chega à Final Answer — e o que o executor enxuto devolve ao parar?

> **💡 Dica:** `intermediate_steps` é uma lista de tuplas `(action, observation)` — e `action.log` traz o bloco Thought/Action logado pelo ReAct.


In [ ]:
# Exercício 3 — limites do executor
!pip install -q langchain langchain-classic langchain-ollama langchain-community chromadb duckduckgo-search

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a09e3", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

prompt_react = hub.pull("hwchase17/react")

@tool
def buscar_nos_documentos(query: str) -> str:
    """Use quando o usuário perguntar sobre informações dos documentos do
    domínio (manuais, contratos, regulamentos). Retorna trechos relevantes
    com número de página. NÃO use para busca na web nem para cálculos."""
    docs = retriever.invoke(query)
    if not docs:
        return "Nenhum documento relevante encontrado."
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

@tool
def buscar_na_web(query: str) -> str:
    """Use quando precisar de informações atuais NÃO presentes nos documentos
    (cotações, notícias, eventos recentes). NÃO use para conteúdo interno
    do domínio do grupo."""
    return DuckDuckGoSearchRun().run(query)

@tool
def calcular(expressao: str) -> str:
    """Use para cálculos matemáticos. Recebe expressão Python válida
    (ex: '24*30'). NÃO use para buscar informações."""
    try:
        return str(eval(expressao, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"

tools = [buscar_nos_documentos, buscar_na_web, calcular]

agente = create_react_agent(llm, tools, prompt_react)

executor_5 = AgentExecutor(agent=agente, tools=tools, verbose=True,
                           max_iterations=___, handle_parsing_errors=True)
executor_2 = AgentExecutor(agent=agente, tools=tools, verbose=True,
                           max_iterations=___, handle_parsing_errors=True)

pergunta_multi = "Qual o prazo de garantia em dias (meses × 30)?"
for nome, ex in [("max_iterations=5", executor_5), ("max_iterations=2", executor_2)]:
    r = ex.invoke({"input": pergunta_multi})
    print(f"\n[{nome}] output: {r['output'][:120]}")
    # 👉 LACUNA 3: o rascunho das iterações fica em resultado["___"]
    for i, (action, obs) in enumerate(r[___], 1):
        print(f"  Iteração {i}: tool={action.tool}, obs_len={len(str(obs))} chars")

# max_iterations=5 → completa: RAG (24 meses) → calcular (24*30) → 720 dias.
# max_iterations=2 → pode parar SEM Final Answer: 'output' traz a mensagem
# de parada e 'intermediate_steps' preserva o rascunho.


### Exercício 4 — Quarta tool: Wikipedia no roster do agente · ★★☆ · 10 min

*Individual · Colab*

1. Instancie a tool pronta `WikipediaQueryRun` completando `top_k_results`.
2. Complete o refinamento da description — quando NÃO usar.
3. Remonte o executor com as 4 tools e rode as 3 perguntas de roteamento.

> **💡 Dica:** tools prontas já vêm com description preenchida — refinar é a forma barata de alinhar o roteamento com as suas outras tools.


In [ ]:
# Exercício 4 — quarta tool no roster
!pip install -q langchain langchain-classic langchain-ollama langchain-community chromadb duckduckgo-search

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a09e4", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

prompt_react = hub.pull("hwchase17/react")

@tool
def buscar_nos_documentos(query: str) -> str:
    """Use quando o usuário perguntar sobre informações dos documentos do
    domínio (manuais, contratos, regulamentos). Retorna trechos relevantes
    com número de página. NÃO use para busca na web nem para cálculos."""
    docs = retriever.invoke(query)
    if not docs:
        return "Nenhum documento relevante encontrado."
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

@tool
def buscar_na_web(query: str) -> str:
    """Use quando precisar de informações atuais NÃO presentes nos documentos
    (cotações, notícias, eventos recentes). NÃO use para conteúdo interno
    do domínio do grupo."""
    return DuckDuckGoSearchRun().run(query)

@tool
def calcular(expressao: str) -> str:
    """Use para cálculos matemáticos. Recebe expressão Python válida
    (ex: '24*30'). NÃO use para buscar informações."""
    try:
        return str(eval(expressao, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"

tools = [buscar_nos_documentos, buscar_na_web, calcular]

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# 👉 LACUNA 1: quantos resultados a tool pronta deve devolver
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=___))
# 👉 LACUNA 2: refine a description — quando NÃO usar
wikipedia.description += " ___"

tools4    = [buscar_nos_documentos, buscar_na_web, calcular, wikipedia]
agente4   = create_react_agent(llm, tools4, prompt_react)
executor4 = AgentExecutor(agent=agente4, tools=tools4, verbose=True,
                          max_iterations=5, handle_parsing_errors=True)
for q in [
    "Qual é a cláusula de garantia no documento?",
    "Quem é o autor de 'Inteligência Artificial' com Norvig?",
    "Quanto é 450 * 1.12?",
]:
    print("\n" + "=" * 50 + f"\n{q}")
    print(executor4.invoke({"input": q})["output"])


## 📚 Referências da aula

- Paper Yao, S. et al. — "ReAct: Synergizing Reasoning and Acting in Language Models." ICLR, 2023. O paper original do padrão ReAct. arxiv.org/abs/2210.03629
- Blog Anthropic Engineering — "Building Effective Agents" (2025). A referência desta aula para quando usar workflow vs. agente. anthropic.com/engineering/building-effective-agents
- Docs LangChain — AgentExecutor, create_react_agent, @tool decorator. python.langchain.com/docs/how_to/agent_executor
- Segurança OWASP LLM Top 10 — LLM01: Prompt Injection. Documentação de riscos de segurança em sistemas LLM. owasp.org/www-project-top-10-for-large-language-model-applications
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2 — Agentes inteligentes: o modelo percepção-ação que fundamenta o loop agêntico desta aula.
- Ebook Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 18: Guardrails/Safety Patterns — as seis camadas de defesa por trás do guardrail de prompt injection desta aula.

---

**Próxima Aula — Aula 10 · 19/10** — Context Engineering para agentes — curadoria em loop agêntico
  
Curar o contexto em loop. RAG como tool oficial. Memory seletiva entre sessões.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*